# Phase 3 — Exploratory Data Analysis

This notebook explores how used-car listing prices relate to vehicle identity, age, usage, specifications, seller type, and market coverage. It reads the cleaned Phase 2 table without modifying it.

Phase 3 does **not** impute missing values, split data, engineer model features, or train a model. Group comparisons are descriptive—not causal conclusions.

## 1. Imports and project paths

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    EDA_REPORT_PATH, EDA_SUMMARY_PATH, FIGURES_DIR,
    PROCESSED_DATA_PATH, TABLES_DIR,
)
from src.eda import create_eda_tables, format_inr, load_cleaned_data, run_eda
from src.validate_data import file_sha256

print(f"Project root: {PROJECT_ROOT}")
print(f"Cleaned input: {PROCESSED_DATA_PATH}")

## 2. Load the cleaned data

A SHA-256 hash identifies the exact Phase 2 input. We check it again at the end to prove that EDA did not change the CSV. The two missing seat values and all 12 extreme odometer records remain untouched.

In [ ]:
cleaned_hash_before = file_sha256(PROCESSED_DATA_PATH)
cars = load_cleaned_data(PROCESSED_DATA_PATH)

print(f"Shape: {cars.shape}")
print(f"Memory: {cars.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"Missing seats left unimputed: {int(cars['seats'].isna().sum())}")
print(f"Rows above 500,000 km retained: {int(cars['km_driven'].gt(500_000).sum())}")
print(f"Input SHA-256: {cleaned_hash_before}")
display(cars.head())

## 3. Price distribution and price bands

The target is strongly right-skewed: a small luxury tail pulls the mean above the median. We therefore emphasize medians and show both raw and log-scale distributions.

In [ ]:
price_summary = cars['selling_price'].describe(percentiles=[0.50, 0.95, 0.99])
print(f"Median: {format_inr(cars['selling_price'].median())}")
print(f"Mean: {format_inr(cars['selling_price'].mean())}")
print(f"Skewness: {cars['selling_price'].skew():.2f}")
display(price_summary.rename('selling_price_inr').to_frame())

eda_tables = create_eda_tables(cars)
display(eda_tables['price_band_summary'])

## 4. Price by brand, model, fuel, transmission, and seller

Counts accompany group medians because a high median based on only a few listings is unstable. Fuel, transmission, and seller gaps also reflect different brands, models, trims, and ages in each group.

In [ ]:
common_brands = eda_tables['brand_price_summary'].nlargest(12, 'count')
common_models = (
    eda_tables['model_price_summary']
    .loc[lambda table: table['count'].ge(100)]
    .nlargest(15, 'median_price')
)

display(common_brands[['brand', 'count', 'median_price', 'q25_price', 'q75_price']])
display(common_models[['brand', 'model', 'count', 'median_price']])
display(eda_tables['category_price_summary'][['category_type', 'category', 'count', 'median_price']])

## 5. Numerical relationships

Spearman correlation measures monotonic association and is less sensitive than Pearson correlation to skew and extreme values. Correlation still does not control for brand, model, segment, or omitted variables.

In [ ]:
pearson_with_price = (
    eda_tables['pearson_correlation']['selling_price']
    .drop('selling_price')
    .sort_values(ascending=False)
    .rename('pearson_with_price')
)
spearman_with_price = (
    eda_tables['spearman_correlation']['selling_price']
    .drop('selling_price')
    .sort_values(ascending=False)
    .rename('spearman_with_price')
)
display(pd.concat([pearson_with_price, spearman_with_price], axis=1))
display(eda_tables['vehicle_age_price_summary'].head(16))

## 6. Coverage and a more comparable seller check

Rare categories will need lower confidence later. For seller type, we also compare Dealer and Individual medians within the same brand-model-age groups, retaining only groups with at least five listings of each type. This reduces—but cannot remove—confounding.

In [ ]:
coverage = eda_tables['category_coverage']
coverage_overview = (
    coverage.groupby('category_type')
    .agg(categories=('category', 'size'), rare_under_50=('is_rare_under_50', 'sum'))
    .sort_index()
)
display(coverage_overview)

seller_matches = eda_tables['matched_seller_comparison']
print(f"Matched groups: {len(seller_matches):,}")
print(f"Median Dealer minus Individual: {format_inr(seller_matches['dealer_minus_individual'].median())}")
print(f"Dealer median higher: {seller_matches['dealer_minus_individual'].gt(0).mean() * 100:.1f}% of groups")
display(seller_matches.head())

## 7. Generate all Phase 3 outputs

The reusable module writes 10 figures, nine CSV summary tables, a JSON metrics file, and a Markdown report. Running this cell again safely replaces only generated Phase 3 outputs.

In [ ]:
eda_summary = run_eda()

print(f"Figures created: {len(eda_summary['outputs']['figures'])}")
print(f"Tables created: {len(eda_summary['outputs']['tables'])}")
print(f"Summary: {EDA_SUMMARY_PATH}")
print(f"Report: {EDA_REPORT_PATH}")

## 8. Review the generated charts

### Price distribution

![Raw and log price distributions](../reports/figures/01_price_distribution.png)

### Brand medians

![Median price by common brand](../reports/figures/02_brand_price_comparison.png)

### Vehicle age and price

![Vehicle age price trend](../reports/figures/05_vehicle_age_price_trend.png)

### Numerical correlation

![Spearman correlation heatmap](../reports/figures/08_spearman_correlation_heatmap.png)

The full visual report at `reports/eda_report.md` links all 10 figures.

## 9. Final verification

These assertions confirm the input stayed unchanged and that Phase 3 produced the complete output set without modeling or imputation.

In [ ]:
cleaned_hash_after = file_sha256(PROCESSED_DATA_PATH)
assert cleaned_hash_after == cleaned_hash_before
assert eda_summary['source']['rows'] == 15_244
assert len(eda_summary['outputs']['figures']) == 10
assert len(eda_summary['outputs']['tables']) == 9
assert eda_summary['modeling_or_imputation_performed'] is False
assert all((PROJECT_ROOT / path).exists() for path in eda_summary['outputs']['figures'])
assert all((PROJECT_ROOT / path).exists() for path in eda_summary['outputs']['tables'])

print("Phase 3 verification passed. The cleaned input is unchanged.")
print("Next: Phase 4 will evaluate candidate features using leakage-safe cross-validation.")